In [2]:

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_lfw_people

SEED = 2027
WINDOW_SIZE = 20

# Reuse the existing embedding CSV if it is uploaded to this runtime.
# Otherwise, upload it when prompted.
import os
from google.colab import files

embedding_file = "lfw_vit_embeddings.csv"

if not os.path.exists(embedding_file):
    print("Please upload lfw_vit_embeddings.csv")
    uploaded = files.upload()

    if embedding_file not in uploaded:
        raise FileNotFoundError(
            "Upload the file named lfw_vit_embeddings.csv"
        )

df = pd.read_csv(embedding_file)

feature_cols = [
    c for c in df.columns if c != "identity_label"
]

X = df[feature_cols].to_numpy(dtype=np.float64)
y = df["identity_label"].to_numpy()

print("Embedding matrix:", X.shape)
print("Identity labels:", len(np.unique(y)))
print("Missing values:", int(df.isna().sum().sum()))
print("All feature values finite:", bool(np.isfinite(X).all()))


Please upload lfw_vit_embeddings.csv


Saving lfw_vit_embeddings.csv to lfw_vit_embeddings.csv
Embedding matrix: (3023, 768)
Identity labels: 62
Missing values: 0
All feature values finite: True


In [3]:

import numpy as np
import pandas as pd

SEED = 2027
WINDOW_SIZE = 20

rng = np.random.default_rng(SEED)

# X and y were loaded in Step 37
reference_idx = []
calibration_idx = []
validation_idx = []

for identity in np.unique(y):
    idx = rng.permutation(np.flatnonzero(y == identity))
    n = len(idx)

    # Allocate approximately 50% / 25% / 25%
    n_ref = int(0.50 * n)
    n_cal = int(0.25 * n)

    reference_idx.extend(idx[:n_ref])
    calibration_idx.extend(idx[n_ref:n_ref + n_cal])
    validation_idx.extend(idx[n_ref + n_cal:])

reference_idx = np.array(reference_idx)
calibration_idx = np.array(calibration_idx)
validation_idx = np.array(validation_idx)

X_reference_raw = X[reference_idx]
X_calibration_raw = X[calibration_idx]
X_validation_raw = X[validation_idx]

# Standardize using reference statistics only
ref_mean_raw = X_reference_raw.mean(axis=0)
ref_std_raw = X_reference_raw.std(axis=0)
ref_std_raw[ref_std_raw < 1e-8] = 1.0

X_reference = (X_reference_raw - ref_mean_raw) / ref_std_raw
X_calibration = (X_calibration_raw - ref_mean_raw) / ref_std_raw
X_validation = (X_validation_raw - ref_mean_raw) / ref_std_raw

# Verify disjoint image indices
assert len(set(reference_idx) & set(calibration_idx)) == 0
assert len(set(reference_idx) & set(validation_idx)) == 0
assert len(set(calibration_idx) & set(validation_idx)) == 0

print("Reference images:", len(reference_idx))
print("Calibration images:", len(calibration_idx))
print("Validation images:", len(validation_idx))

print("Reference identities:", len(np.unique(y[reference_idx])))
print("Calibration identities:", len(np.unique(y[calibration_idx])))
print("Validation identities:", len(np.unique(y[validation_idx])))

print("Image overlap between pools: 0")
print("All standardized features finite:",
      np.isfinite(X_reference).all()
      and np.isfinite(X_calibration).all()
      and np.isfinite(X_validation).all())


Reference images: 1497
Calibration images: 737
Validation images: 789
Reference identities: 62
Calibration identities: 62
Validation identities: 62
Image overlap between pools: 0
All standardized features finite: True


In [4]:

import numpy as np
import pandas as pd

WINDOW_SIZE = 20
TARGET_FAR = 0.01
SEED = 2027

rng = np.random.default_rng(SEED)

# Create non-overlapping windows from each pool independently.
def make_windows(X_pool, window_size, rng):
    X_pool = X_pool[rng.permutation(len(X_pool))]
    n = (len(X_pool) // window_size) * window_size
    return X_pool[:n].reshape(-1, window_size, X_pool.shape[1])

reference_windows = make_windows(
    X_reference, WINDOW_SIZE, rng
)
calibration_windows = make_windows(
    X_calibration, WINDOW_SIZE, rng
)
validation_windows = make_windows(
    X_validation, WINDOW_SIZE, rng
)

# Estimate reference statistics in standardized feature space
reference_mean = X_reference.mean(axis=0)

# Regularized covariance for Mahalanobis distance
covariance = np.cov(X_reference, rowvar=False)
ridge = 1e-3
covariance_reg = covariance + ridge * np.eye(covariance.shape[0])
inverse_covariance = np.linalg.pinv(covariance_reg)

def detector_scores(windows):
    means = windows.mean(axis=1)
    differences = means - reference_mean

    euclidean = np.linalg.norm(differences, axis=1)

    mahalanobis_squared = np.einsum(
        "bi,ij,bj->b",
        differences,
        inverse_covariance,
        differences
    )
    mahalanobis = np.sqrt(np.maximum(mahalanobis_squared, 0))

    return {
        "Euclidean": euclidean,
        "Mahalanobis": mahalanobis
    }

calibration_scores = detector_scores(calibration_windows)
validation_scores = detector_scores(validation_windows)

# Calibrate thresholds only on the calibration pool
thresholds = {}

for detector, scores in calibration_scores.items():
    thresholds[detector] = np.quantile(
        scores,
        1 - TARGET_FAR,
        method="higher"
    )

# Evaluate only once on the independent validation pool
validation_results = []

for detector, scores in validation_scores.items():
    alarms = scores > thresholds[detector]

    validation_results.append({
        "detector": detector,
        "threshold": float(thresholds[detector]),
        "calibration_windows": len(calibration_scores[detector]),
        "validation_windows": len(scores),
        "validation_false_alarms": int(alarms.sum()),
        "validation_false_alarm_rate": float(alarms.mean())
    })

validation_summary = pd.DataFrame(validation_results)

print("Reference windows:", len(reference_windows))
print("Calibration windows:", len(calibration_windows))
print("Validation windows:", len(validation_windows))
display(validation_summary.round(4))


Reference windows: 74
Calibration windows: 36
Validation windows: 39


,detector,threshold,calibration_windows,validation_windows,validation_false_alarms,validation_false_alarm_rate
0,Euclidean,8.4724,36,39,0,0.0
1,Mahalanobis,14.3602,36,39,0,0.0


In [5]:

# Save the actual independent validation results
validation_summary.to_csv(
    "lfw_independent_threshold_validation.csv",
    index=False
)

print("Saved: lfw_independent_threshold_validation.csv")

from google.colab import files
files.download("lfw_independent_threshold_validation.csv")


Saved: lfw_independent_threshold_validation.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:

import pandas as pd
from math import sqrt

# Summarize the current independent validation sample
validation_summary = pd.read_csv(
    "lfw_independent_threshold_validation.csv"
)

for _, row in validation_summary.iterrows():
    n = int(row["validation_windows"])
    alarms = int(row["validation_false_alarms"])

    # Approximate 95% upper bound when zero alarms are observed
    upper_bound = 1 - (0.05 ** (1 / n)) if alarms == 0 else None

    print(f"\nDetector: {row['detector']}")
    print(f"Validation windows: {n}")
    print(f"Observed false alarms: {alarms}")
    print(f"Observed false-alarm rate: {alarms / n:.4f}")

    if upper_bound is not None:
        print(
            "Approximate 95% upper bound on false-alarm probability:",
            f"{upper_bound:.4f}"
        )



Detector: Euclidean
Validation windows: 39
Observed false alarms: 0
Observed false-alarm rate: 0.0000
Approximate 95% upper bound on false-alarm probability: 0.0739

Detector: Mahalanobis
Validation windows: 39
Observed false alarms: 0
Observed false-alarm rate: 0.0000
Approximate 95% upper bound on false-alarm probability: 0.0739


In [7]:

# Inspect the available independent validation pool

print("Validation images:", len(X_validation))
print("Feature dimensions:", X_validation.shape[1])
print("Window size:", WINDOW_SIZE)
print(
    "Maximum non-overlapping windows:",
    len(X_validation) // WINDOW_SIZE
)

print(
    "Images remaining after complete windows:",
    len(X_validation) % WINDOW_SIZE
)

print(
    "Minimum images needed for 299 windows:",
    299 * WINDOW_SIZE
)


Validation images: 789
Feature dimensions: 768
Window size: 20
Maximum non-overlapping windows: 39
Images remaining after complete windows: 9
Minimum images needed for 299 windows: 5980


In [8]:

from pathlib import Path
import pandas as pd

validation_summary = pd.read_csv(
    "lfw_independent_threshold_validation.csv"
)

report = """# LFW Independent Threshold Validation

## Objective
Evaluate Euclidean and regularized Mahalanobis drift scores using separate
reference, threshold-calibration, and no-drift validation image pools.

## Dataset and split
- Dataset: Labeled Faces in the Wild (LFW)
- Extractor: google/vit-base-patch16-224
- Features: 768-dimensional generic ViT image features
- Reference images: 1,497
- Calibration images: 737
- Validation images: 789
- Window size: 20 images
- Reference windows: 74
- Calibration windows: 36
- Validation windows: 39
- Image-index overlap between pools: 0

The extractor is a general-purpose vision model, not a validated face-recognition
model. The experiment therefore evaluates feature-distribution drift, not
validated biometric identity drift.

## Threshold calibration
The reference pool was used to estimate feature statistics. Detector thresholds
were selected using the separate calibration pool at the empirical 99th
percentile. The validation pool was not used to choose thresholds.

## No-drift validation results

| Detector | Threshold | Validation windows | False alarms | Observed false-alarm rate |
|---|---:|---:|---:|---:|
"""

for _, row in validation_summary.iterrows():
    report += (
        f"| {row['detector']} "
        f"| {row['threshold']:.4f} "
        f"| {int(row['validation_windows'])} "
        f"| {int(row['validation_false_alarms'])} "
        f"| {row['validation_false_alarm_rate']:.4f} |\n"
    )

report += """
## Statistical uncertainty
Both detectors produced zero observed false alarms in 39 validation windows.
The approximate one-sided 95% binomial upper bound for the false-alarm
probability is 7.39% when zero alarms are observed in 39 independent trials.

This bound depends on an independence assumption. Windows formed from images
in the same dataset may not be fully independent, particularly because identity
composition and image acquisition conditions can be related.

The results do not establish a 1% false-alarm guarantee. Only 39 non-overlapping
validation windows were available.

## Limitations
1. LFW does not provide a natural chronological biometric stream.
2. The feature extractor is a generic ViT rather than a validated face-recognition model.
3. The validation sample is too small to certify a low false-alarm target.
4. Identity composition and acquisition conditions may affect feature distributions.
5. Zero observed false alarms does not imply a zero underlying false-alarm probability.

## Conclusion
This is a preliminary independent-threshold-calibration exercise on real-image
features. The observed false-alarm rate was zero for both detectors in the
available validation sample, but uncertainty remains substantial. A larger,
carefully designed follow-up study is required before making stronger claims.

## Reproducibility
Notebook: 07_LFW_Independent_Threshold_Validation.ipynb

Results file: lfw_independent_threshold_validation.csv
"""

Path("LFW_Independent_Threshold_Validation_Report.md").write_text(
    report, encoding="utf-8"
)

print("Saved: LFW_Independent_Threshold_Validation_Report.md")
print(report)


Saved: LFW_Independent_Threshold_Validation_Report.md
# LFW Independent Threshold Validation

## Objective
Evaluate Euclidean and regularized Mahalanobis drift scores using separate
reference, threshold-calibration, and no-drift validation image pools.

## Dataset and split
- Dataset: Labeled Faces in the Wild (LFW)
- Extractor: google/vit-base-patch16-224
- Features: 768-dimensional generic ViT image features
- Reference images: 1,497
- Calibration images: 737
- Validation images: 789
- Window size: 20 images
- Reference windows: 74
- Calibration windows: 36
- Validation windows: 39
- Image-index overlap between pools: 0

The extractor is a general-purpose vision model, not a validated face-recognition
model. The experiment therefore evaluates feature-distribution drift, not
validated biometric identity drift.

## Threshold calibration
The reference pool was used to estimate feature statistics. Detector thresholds
were selected using the separate calibration pool at the empirical 99th

In [9]:
from google.colab import files
files.download("LFW_Independent_Threshold_Validation_Report.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>